In [ ]:
from fmsae_repro import LEGACY_ANALYSIS, load_manifest
import ezslide
import lazyslide as zs
import mesoslide as ms
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
manifest = load_manifest()
slides = ezslide.open_slides(manifest, attach_images=False)

In [ ]:
file_path = f'{LEGACY_ANALYSIS}/SAE_paper/TB/3D_HnE/relu_sae/reordered_active_features.csv'
df = pd.read_csv(file_path, index_col=0)
df.dropna(inplace=True)
df['revised_cluster'] = df['revised_cluster'].astype(int)

feature_to_group = dict(zip(df['feature_idx'].astype(int), df['revised_cluster']))

_ = ms.tl.aggregate_feature_groups(
    slides,
    feature_prefix='UNI_SAE',
    feature_to_group=feature_to_group,
    method='max',
)


In [ ]:
patches = ms.select_top_patches(slides, 'group_21', 20, top_fraction=0.1)
ms.plotting.plot_patch_gallery(patches, slides, return_fig=True)

In [ ]:
patches = ms.select_top_patches(slides, 'group_21', 1000, top_fraction=1.0)
n_tokens = (patches.obsm['UNI_knn_labels'] == 12).sum(axis=1)
scores = patches.obs['group_21'].values
n_tokens, scores

In [ ]:
# plot correlation as well
corr = pd.DataFrame({'n_tokens': n_tokens, 'scores': scores}).corr().iloc[0, 1]
plt.scatter(n_tokens/196, scores)
plt.title(f'Correlation: {corr:.2f}')

## Reviewer Comment 1: quantitative validation of the feature-group ↔ token-class correspondence

1. **Between-group distinguishability**: for each feature group, sample the top-scoring patches and compare their token-class composition against every other group (one-vs-rest Mann-Whitney U, FDR-corrected), rendered as a log2 fold-change heatmap.
2. **Within-group reproducibility**: for each feature group, sample patches spread evenly across its whole score range and correlate the group's own score against each token-class fraction (Spearman, FDR-corrected), rendered as a correlation heatmap.
3. **Targeted examples**: boxplots for the reviewer's own examples (e.g. MNGC, lymphoid aggregate, granuloma necrotic core), comparing a reference group against a curated set of known-negative groups with pairwise significance brackets.

In [ ]:
import numpy as np
import seaborn as sns
from scipy.stats import mannwhitneyu, kruskal, spearmanr
from statsmodels.stats.multitest import multipletests
from statannotations.Annotator import Annotator

feature_group_names = {
    1: 'Pleura',
    2: 'Respiratory epithelium',
    3: 'Blood vessel intima/media',
    4: 'Blood vessel adventitia',
    5: 'Atelectatic lung parenchyma',
    6: 'Interstitial inflammation',
    7: 'Transitional zone',
    8: 'Intra-alveolar macrophages',
    9: 'Capillary congestion',
    10: 'Normal alveolar wall',
    11: 'AW with interstitial fibrosis',
    12: 'AW with interstitial inflammation',
    13: 'AW with intra-alveolar RBCs',
    14: 'Small BV (arteriole or venule)',
    15: 'Anthracosis',
    16: 'Lymphoid aggregate',
    17: 'Loose connective tissue',
    18: 'Dense collagenous tissue',
    19: 'Elastosis',
    20: 'DCI with inflammation',
    21: 'Multinucleated giant cell',
    22: 'Epithelial histiocytes',
    23: 'Fibrotic wall of granuloma',
    24: 'Fibrolymphoplasmacytic wall',
    25: 'Periphery of granuloma necrotic core',
    26: 'Center of granuloma necrotic core',
    27: 'Red blood cells',
    28: 'Blank space (or debris)',
    29: 'Miscellaneous',
}

# NOTE: a "Background" class was inserted at index 0, and the three entries near
# the end previously had no commas between them, which Python's implicit
# string-literal concatenation would have silently merged into one element --
# fixed here so the list has exactly 21 entries (indices 0-20), aligned
# positionally with the integer class ids in obsm['UNI_knn_labels'].
token_label_names = [
    "Background",                              # 0
    "Air + debris + tissue edge",             # 1
    "RBC",                                     # 2
    "Alveolar wall",                           # 3
    "Respiratory epithelium",                  # 4
    "Blood vessel",                            # 5
    "Alveolar macs",                           # 6
    "Reactive pneumocytes",                    # 7
    "Fibrosis around pleura/blood vessel",     # 8
    "Alveolar fibrosis",                       # 9
    "Dense fibrosis around granuloma",         # 10
    "Anthracosis",                             # 11
    "MNGC",                                    # 12
    "Epithelioid histiocytes",                 # 13
    "Lymphocyte aggregate",                    # 14
    "Lymphocytes (plasma cells?)",             # 15
    "Fibrosis associated immune infiltrate",   # 16
    "Granuloma associated immune infiltrate",  # 17
    "Pleura mesothelium",                      # 18
    "Cellular border of necrosis",             # 19
    "Necrotic core",                           # 20
]

group_ids = list(range(1, 29))  # 1-28, excludes group 29 ("Miscellaneous")

In [ ]:
def fdr_correct_ignoring_nan(pvals):
    """BH-correct only the finite p-values; untestable (nan) cells stay nan."""
    flat = np.asarray(pvals, dtype=float).ravel()
    qvals = np.full_like(flat, np.nan)
    valid = np.isfinite(flat)
    if valid.any():
        _, qvals[valid], *_ = multipletests(flat[valid], method="fdr_bh")
    return qvals.reshape(np.shape(pvals))


def token_class_fractions(patches, token_key="UNI_knn_labels", n_classes=21):
    """(n_patches, n_classes) fraction of a patch's 196 tokens in each class."""
    labels = patches.obsm[token_key].astype(int)  # (n_patches, 196)
    one_hot = labels[:, :, None] == np.arange(n_classes)[None, None, :]
    return one_hot.mean(axis=1)  # fraction, not count


def between_group_analysis(slides, group_ids, n=1000, top_fraction=0.1,
                            token_key="UNI_knn_labels", n_classes=21):
    """Between-group distinguishability: one-vs-rest Mann-Whitney U per (group, token) cell.

    Unit of analysis is the patch, not the token -- the 196 tokens inside one
    patch are spatially correlated, so treating them as independent draws
    would be pseudoreplication and overstate significance.
    """
    per_group_fracs = {}
    for g in group_ids:
        patches = ms.select_top_patches(slides, f"group_{g}", n, top_fraction=top_fraction)
        per_group_fracs[g] = token_class_fractions(patches, token_key, n_classes)  # (n, n_classes)

    mean_matrix = np.vstack([per_group_fracs[g].mean(axis=0) for g in group_ids])  # (G, n_classes)
    global_mean = mean_matrix.mean(axis=0)  # (n_classes,)
    log2fc = np.log2((mean_matrix + 1e-4) / (global_mean + 1e-4))

    # Cell-level test: one-sided Mann-Whitney U, group g's n values vs. pooled others.
    pvals = np.ones((len(group_ids), n_classes))
    for i, g in enumerate(group_ids):
        others = np.concatenate([per_group_fracs[g2] for g2 in group_ids if g2 != g], axis=0)
        for k in range(n_classes):
            _, pvals[i, k] = mannwhitneyu(per_group_fracs[g][:, k], others[:, k],
                                           alternative="greater")
    qvals = fdr_correct_ignoring_nan(pvals)

    # Secondary summary: omnibus Kruskal-Wallis per token class (any between-group difference at all).
    kw_p = [kruskal(*[per_group_fracs[g][:, k] for g in group_ids])[1] for k in range(n_classes)]
    kw_q = fdr_correct_ignoring_nan(np.array(kw_p))

    return log2fc, qvals, kw_q, per_group_fracs


def within_group_analysis(slides, group_ids, n=1000, token_key="UNI_knn_labels", n_classes=21):
    """Within-group reproducibility: Spearman(group score, token fraction) per (group, token) cell.

    Samples spread evenly across the whole non-negative score range
    (top_fraction=1.0), not just the top-scoring patches, so there is a
    meaningful spread of scores to correlate against.
    """
    rho = np.zeros((len(group_ids), n_classes))
    pvals = np.ones_like(rho)
    for i, g in enumerate(group_ids):
        patches = ms.select_top_patches(slides, f"group_{g}", n, top_fraction=1.0)
        frac = token_class_fractions(patches, token_key, n_classes)
        scores = patches.obs[f"group_{g}"].to_numpy()
        for k in range(n_classes):
            rho[i, k], pvals[i, k] = spearmanr(scores, frac[:, k])
    qvals = fdr_correct_ignoring_nan(pvals)
    return rho, qvals


def plot_annotated_heatmap(matrix, qvals, row_labels, col_labels, cmap="RdBu_r",
                            center=0, title=""):
    matrix = np.asarray(matrix, dtype=float)
    qvals = np.asarray(qvals, dtype=float)
    untestable = np.isnan(qvals)  # constant input -> no test could be run for this cell

    annot = np.full(matrix.shape, "", dtype=object)
    annot[qvals < 0.05] = "*"
    annot[qvals < 0.01] = "**"
    annot[qvals < 0.001] = "***"

    fig, ax = plt.subplots(figsize=(0.5 * len(col_labels) + 2, 0.35 * len(row_labels) + 2))
    ax.set_facecolor("lightgray")  # shows through wherever the heatmap is masked
    sns.heatmap(matrix, mask=untestable, annot=annot, fmt="", cmap=cmap, center=center,
                xticklabels=col_labels, yticklabels=row_labels, ax=ax,
                linewidths=0.4, linecolor="white")

    # Hatch the untestable cells so "not testable" is visually distinct from
    # "tested, not significant" (which just gets a near-zero, near-white color).
    for i, j in zip(*np.where(untestable)):
        ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, hatch="///",
                                    edgecolor="dimgray", linewidth=0))

    ax.set_title(title)
    # ax.text(0.0, -0.14,
    #         "Hatched: untestable (constant input, e.g. token absent from every sampled patch)",
    #         transform=ax.transAxes, fontsize=8, color="dimgray")
    fig.tight_layout()
    return fig, ax

In [ ]:
def targeted_group_comparison(slides, token_to_groups, n=1000, top_fraction=0.1,
                               token_key="UNI_knn_labels", n_classes=21,
                               feature_group_names=None, token_label_names=None):
    """One boxplot per (token, [ref_group, *comparison_groups]) entry.

    The reference group (first in the list) is compared against every other
    group in the list via a one-sided Mann-Whitney U test (reference expected
    higher), FDR-corrected (BH) within each plot's small set of comparisons,
    with significance brackets drawn from the reference box to each other box.
    """
    figs = {}
    for token_id, ids in token_to_groups.items():
        if len(ids) < 2:
            raise ValueError(f"token {token_id}: need at least 2 groups (reference + comparisons).")

        records = []
        for g in ids:
            patches = ms.select_top_patches(slides, f"group_{g}", n, top_fraction=top_fraction)
            frac = token_class_fractions(patches, token_key, n_classes)[:, token_id]
            label = feature_group_names[g] if feature_group_names else str(g)
            records.extend({"group": label, "fraction": v} for v in frac)
        df = pd.DataFrame(records)
        order = [feature_group_names[g] if feature_group_names else str(g) for g in ids]

        n_comparisons = len(order) - 1
        fig, ax = plt.subplots(figsize=(1.3 * len(ids) + 2, 5.0 + 0.8 * n_comparisons))
        palette = ["#d95f02"] + ["#7570b3"] * n_comparisons  # reference group stands out
        sns.boxplot(data=df, x="group", y="fraction", order=order, showfliers=False,
                    palette=palette, width=0.55, ax=ax)
        sns.stripplot(data=df, x="group", y="fraction", order=order, ax=ax,
                       color="black", size=3, alpha=0.4, jitter=0.15)
        fig.tight_layout()  # settle box/strip/label spacing before the annotator adds brackets

        pairs = [(order[0], o) for o in order[1:]]  # reference vs. each other group
        annotator = Annotator(ax, pairs, data=df, x="group", y="fraction", order=order)
        annotator.configure(test="Mann-Whitney-gt", text_format="star", loc="inside",
                             comparisons_correction="BH", verbose=0)
        annotator.apply_and_annotate()

        token_name = token_label_names[token_id] if token_label_names else f"token {token_id}"
        # fig.suptitle(f"{token_name} fraction: {order[0]} vs. comparison groups (n={n} patches/group)",
        #              y=0.98, fontsize=11)
        ax.set_xlabel("")
        ax.set_ylabel(f"{token_name} token fraction (per patch)")
        plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
        # Reserve headroom for the suptitle above the brackets statannotations just
        # added -- this must be the LAST layout call: tight_layout() would recompute
        # margins from scratch and silently undo this reservation if called after.
        # fig.subplots_adjust(top=0.8)
        # figs[token_id] = fig
    return figs

In [ ]:
# Analysis 1: between-group distinguishability
log2fc, between_qvals, kw_q, per_group_fracs = between_group_analysis(
    slides, group_ids, n=1000, top_fraction=0.1,
)

row_labels = [feature_group_names[g] for g in group_ids]

sns.set_theme(font_scale=1)
sns.set_style("white")
plot_annotated_heatmap(
    log2fc, between_qvals, row_labels, token_label_names,
    title="Between-group distinguishability: log2FC(group mean vs. global mean), * = FDR q<0.05",
)

print("Omnibus Kruskal-Wallis FDR q-value per token class (any between-group difference at all):")
for name, q in zip(token_label_names, kw_q):
    print(f"  {name}: q={q:.2e}")

In [ ]:
# Analysis 2: within-group reproducibility
rho, within_qvals = within_group_analysis(slides, group_ids, n=1000)
sns.set_theme(font_scale=1)
sns.set_style("white")
plot_annotated_heatmap(
    rho, within_qvals, row_labels, token_label_names,
    title="Within-group reproducibility: Spearman(group score, token fraction), * = FDR q<0.05",
)

In [ ]:
# Analysis 1b: targeted group-vs-group boxplots for the reviewer's own examples.
# Key = token class id, value = [reference_group, *known_negative_groups].
# Adjust the known-negative groups below to your own domain judgment -- these are
# illustrative picks (groups without the structure in question).
token_to_groups = {
    12: [21, 5, 10, 27],  # MNGC -> group 21 (Multinucleated giant cell) vs. known negatives
    14: [16, 5, 10, 27],  # Lymphocyte aggregate -> group 16 (Lymphoid aggregate) vs. known negatives
    # 19: [25, 5, 10, 27],  # Cellular border of necrosis -> group 25 (Periphery of granuloma necrotic core)
    # 20: [26, 5, 10, 27],  # Necrotic core -> group 26 (Center of granuloma necrotic core)
}
sns.set_theme(font_scale=1.5)
sns.set_style("white")
targeted_figs = targeted_group_comparison(
    slides, token_to_groups, n=1000, top_fraction=0.1,
    feature_group_names=feature_group_names, token_label_names=token_label_names,
)